# Das Perzeptron

Mit künstlichen neuronalen Netzwerken werden wir eine neue Form der Datenverarbeitung kennenlernen. An die Stelle einer __zentralen Recheneinheit__, der __CPU__, die durch ein Programm gesteuert Daten verarbeitet (sog. __Von-Neumann-Architektur__, s. [Wikipedia]()), tritt eine Vielzahl sehr viel einfacherer Recheneinheiten, die mit einem einfachen __internen Algorithmus__ arbeiten, und bei denen die komplexe Funktionalität erst durch die __Vernetzung__ dieser Recheneinheiten entsteht. Das Netz wird dabei nicht mehr programmiert, sondern lernt von selbst.

### Das Vorbild: Die Natur

Vorbild für neuronale Netzwerke sind neurowissenschaftliche Erkenntnisse, die in der Mitte des 20. Jahrhunderts gemacht wurden. Die Gehirne von Menschen und Tieren ist aus einer Vielzahl spezialisierter Zellen zusammengesetzt, den sogenannten Neuronen. Ein Neuron besteht wie jede Körperzelle aus einem Zellkörper; beim Neuron ragen aus diesem Körper zahlreiche Fäden heraus, die wiederum mit anderen Neuronen Kontakt haben. Diese __Dendriten__ sind recht dünn, es gibt jedoch eine Art Hauptstrang, das __Axon__, der etwas stärker ausgeprägt ist. Diese Fäden sind Nervenbahnen, längs denen elektrische Impulse laufen, die durch Umklappen elektrochemischer Potentiale gebildet werden. Dabei laufen die elektrischen Impulse durch die Dendriten in das Neuron hinein (über die __Synapsen__), wohingegen das Neuron über das Axon ein eigenes elektrisches Signal ausgibt, das von anderen Neuronen empfangen und verarbeitet wird. 

Die elektrischen Signale haben dabei immer dieselbe Stärke, d.h. ein Neuron sendet entweder ein Signal aus (es _feuert_, oder: es _ist aktiv_), oder es sendet kein Signal aus. Dabei hängt die Aktivität des Neurons von den eingegangenen Signalen ab; ab einem bestimmten Schwellenwert sendet das Neuron ein Signal aus.

### Übertragung in die Computerwelt:  McCulloch-Pitts-Neuronen

Zur selben Zeit machten sich Computer-Experten Gedanken um Alternativen zur oben beschriebenen Von-Neumann-Architektur. Was in der Natur funktionierte, sollte auch in der Technik funktionieren, und die Bauteile wären sehr einfach zu realisieren.

> Eine McCulloch-Pitts-Zelle oder ein __McCulloch-Pitts-Neuron__ ist ein von Warren McCulloch und Walter Pitts im Jahr 1943 vorgeschlagenes Neuronenmodell. Beide wollten ein vereinfachtes Modell realer Vorgänge in neuronalen Strukturen entwerfen, um zu klären, ob das Gehirn die Turing-berechenbaren Funktionen wirklich berechnen kann. Das McCulloch-Pitts-Neuronenmodell ist das einfachste Neuronenmodell der Neuroinformatik überhaupt. Künstliche neuronale Netze aus McCulloch-Pitts-Zellen können ausschließlich binäre Signale verwenden. Jedes einzelne Neuron kann als Ausgabe nur eine 1 oder 0 erzeugen. Analog zu biologischen neuronalen Netzen können hemmende Signale bearbeitet werden. Jede McCulloch-Pitts-Zelle besitzt eine beliebige reelle Zahl als Schwellenwert. [[Quelle: Wikipedia](https://de.wikipedia.org/wiki/McCulloch-Pitts-Zelle) ]

### Eine Python-Implementierung der Aktivierungsfunktion

Wir wollen ein solches künstliches Neuron einmal in Python implementieren. Das einfachste Neuron besitzt zwei Eingabe-Kanäle und einen Ausgabe-Kanal. Jeder binäre Eingabe-Kanal kann dabei auf 0 oder 1 gesetzt werden. Das Neuron verarbeitet diese beiden Signale und setzt dann den eigenen Ausgabe-Kanal auf 0 oder 1. Die Verarbeitung im Neuron geschieht folgendermaßen: Die beiden Eingabekanäle $x_1$ und $x_2$ verfügen über je ein Gewicht $w_1$ und $w_2$, die die Bedeutung des jeweiligen Kanals bestimmen. Diese Gewichte können positiv, also verstärkend, aber auch negativ, also hemmend, sein. Es wird nun die gewichtete Summe $w_1 x_1 + w_2 x_2$ berechnet. Man beachte, dass $x_1$ und $x_2$ entweder 0 oder 1 sind. Liegt diese Summe über einem gegebenen Schwellenwert $s$, dem Aktiverungs-Potential, so setzt das Neuron seinen Ausgabewert $o$ auf 1 (es _feuert_), ansonsten auf 0. Die Gewichte und der Schwellenwert sind dabei beliebige reelle Zahlen.

Mathematisch formuliert: Gilt 

$$ w_1 \, x_1 + w_2 \, x_2 > s,$$

so ist das Neuron aktiv. Diese Aktivierungsfunktion lässt sich in Python leicht erzeugen. Dabei fassen wir die Ein-und Ausgabewerte als boolesche Werte auf; dabei verwenden wir, dass Python True als 1 und False als 0 auffasst:

In [1]:
from typing import Callable

def activation(w1: float,w2: float, s: float) -> Callable[[bool, bool],bool]:
    return lambda x1, x2: w1 * x1 + w2 * x2 > s

Wie wirkt nun die Aktivierungsfunktion auf die Eingabedaten? Dazu definieren wir eine kleine Funktion zur Ausgabe aller möglichen Ergebnisse in einer Wahrheitstafel. Als Beispiel geben wir die Ergebnisse der UND-Verknüpfung aus:

In [2]:
boolean = (True, False)

def wahrheitstabelle(lf,name = ""):
    print(f'   a     b   |  {name}' )
    print(f'-------------|---------')
    for a, b in [ (a,b) for a in boolean for b in boolean]:
        print(f' {a!s:5} {b!s:5} | {lf(a,b)!s}')

wahrheitstabelle(lambda a,b : a and b, "a & b")

   a     b   |  a & b
-------------|---------
 True  True  | True
 True  False | False
 False True  | False
 False False | False


Nun setzen wir statt der UND-Verknüpfung unsere Aktivierunsfunktion ein. Wir wählen als Gewichte und Schwellenwert jeweils die 1:

In [3]:
af = activation(1, 1, 1)
wahrheitstabelle(af)

   a     b   |  
-------------|---------
 True  True  | True
 True  False | False
 False True  | False
 False False | False


Wir erkennen, dass dies dieselbe Wahrheitstafel wie für die UND-Verknüpfung ist: Das Neuron mit den angegebenen Gewichten und Schwellenwert ist genau dann aktiv, wenn beide Eingabe-Kanäle aktiv sind. 

### Implementierung als Klasse

Wir wollen nun das McCulloch-Pitts-Neuron als Python-Klasse implementieren. Der Vorteil zur einfachen Aktivierungsfunktion besteht darin, dass die Gewichte und der Schwellwert  als Objektvariablen gespeichert und verändert werden können.

So sieht nun ein künstliches McCulloch-Pitts-Neuron als Klasse in Python aus:

In [4]:
class McCullochPittsNeuron:
    def __init__(self, w1: float, w2: float, s: float) -> None:
        self.w1 = w1
        self.w2 = w2
        self.s = s
    def activation(self, x1: bool, x2: bool) -> bool: 
        return self.w1 * x1 + self.w2 * x2 > self.s

    def __str__(self):
        return "w1 = {}, w2 = {}, s = {}".format(self.w1,self.w2,self.s)        

Wir speichern die Gewichte also als Objektvariablen, die wir später noch verändern können. Testen wir unser Neuron wierder mit unseren Werten: Gewichte = 1 und Schwellenwert = 1:

In [5]:
neuron = McCullochPittsNeuron(1,1,1)
wahrheitstabelle(neuron.activation)

   a     b   |  
-------------|---------
 True  True  | True
 True  False | False
 False True  | False
 False False | False


#### Wir _programmieren_ ein Neuron

Das McCulloch-Pitts-Neuron lässt sich als __logisches Gatter__ verwenden. Versuchen wir einmal, die Gewichte für ein ODER-Gatter zu bestimmen, d.h. die Eingabewerte (0,0) sollen eine 0 ergeben, und alle anderen Kombinationen eine 1. Bei der Eingabe von (0,0) lautet die gewichtete Summe immer 0, und setzen wir den Schwellenwert $s=0$, so bleibt das Neuron inaktiv. Nun soll das Neuron aktiv sein, wenn einer der beiden Eingabe-Kanäle oder sogar beide aktiv sind. Dazu muss die gewichtete Summe nur positiv sein. Setzen wir also $w_1, w_2 = 1$, so sollte das Neuron als ODER-Gatter funktionieren. Probieren wir es aus:

In [6]:
neuron = McCullochPittsNeuron(1,1,0)
wahrheitstabelle(neuron.activation)

   a     b   |  
-------------|---------
 True  True  | True
 True  False | True
 False True  | True
 False False | False


Die Oder-Verknüpfung ist also korrekt programmiert.

Mit dem folgenden Neuron erhalten wir ein __NAND-Gatter__:

In [7]:
neuron = McCullochPittsNeuron(-1,-1,-2)
wahrheitstabelle(neuron.activation)

   a     b   |  
-------------|---------
 True  True  | False
 True  False | True
 False True  | True
 False False | True


Aus der elementaren Informatik ist bekannt, dass sich aus NAND-Gattern jede Art logischer Schaltung erzeugen lässt, dh. mit diesen Neuronen lassen sich vollständige Computer bauen. Dabei besteht die _Hardware_ aus einer Vielzahl einfacher Bausteine, die jeweils nur eine gewichtete Summe berechnen und diese mit einem Schwellwert vergleichen können. Es gibt keine zentrale Recheneinheit, und die _Programmierung_ des Computers erfolgt durch passende Wahl der Gewichte und Schwellenwerte.

### Neuronen programmieren sich selbst

In der Natur gibt es keinen Programmierer, der die Gewichte und Schwellwerte festsetzt. Stattdessen sind Neuronen in einem Netzwerk in der Lage, ihr Verhalten anzupassen, d.h. im Lauf der Zeit auf eingehende Signale anders zu reagieren: Das gesamte natürliche neuronale Netzwerk __lernt__.

#### Die Hebbsche Lernregel

Der Psychologe __Donald Olding Hebb__ formulierte 1949 in seinem Buch _The Organization of Behavior_: 

> Wenn ein Axon der Zelle A […] Zelle B erregt und wiederholt und dauerhaft zur Erzeugung von Aktionspotentialen in Zelle B beiträgt, so resultiert dies in Wachstumsprozessen oder metabolischen Veränderungen in einer oder in beiden Zellen, die bewirken, dass die Effizienz von Zelle A in Bezug auf die Erzeugung eines Aktionspotentials in B größer wird.“ [[Quelle: Wikipedia](https://de.wikipedia.org/wiki/Hebbsche_Lernregel)]

In künstlichen neuronalen Netzwerken wird dies als __Hebbsche Lernregel__ ($\rightarrow$ [Wikipedia](https://de.wikipedia.org/wiki/Hebbsche_Lernregel)) bezeichnet und gilt als Grundregel für Lernen und Gedächtnis in der Natur. Sie lautet in ihrer einfachsten Form $$\Delta w_i = \eta\,x_i\;o$$

wobei $\Delta w_i$ die Korrektur des Gewichts $w_i$ ist, $x_i$ der Eingabe-Wert und $o$ der Ausgabewert ist. $\eta$ ist eine Konstante, die die Lernrate steuert. I.d.R. ist $\eta \ll 1$.

Die Einführung einer Lernregel führt zu selbstlernenden künstlichen neuronalen Netzwerken, deren erster und einfachster Vertreter 1957 vorgestellt wurde.

### Das Perzeptron

> Das Perzeptron (nach engl. perception, „Wahrnehmung“) ist ein vereinfachtes künstliches neuronales Netz, das zuerst von Frank Rosenblatt 1957 vorgestellt wurde. Es besteht in der Grundversion (einfaches Perzeptron) aus einem einzelnen künstlichen Neuron mit anpassbaren Gewichtungen und einem Schwellenwert. Unter diesem Begriff werden heute verschiedene Kombinationen des ursprünglichen Modells verstanden, dabei wird zwischen einlagigen und mehrlagigen Perzeptren (engl. multi-layer perceptron, MLP) unterschieden. Perzeptron-Netze wandeln einen Eingabevektor in einen Ausgabevektor um und stellen damit einen einfachen Assoziativspeicher dar. [[Wikipedia](https://de.wikipedia.org/wiki/Perzeptron)]

#### Schwellenwert und Bias

Für die Formulierung des Perzeptrons ist es bequem, eine kleine Änderung gegenüber den McCulloch-Pitts-Neuronen vorzunehmen. An die Stelle der Aktivierungsregel 

$$ w_1 x_1 + w_2  x_2 > s $$ 

setzen wir die __Perzeptron-Regel__

$$ w_1 x_1 + w_2 x_2 + b > 0 $$ 

Wir ersetzen also den Schwellenwert $s$ durch den Wert $b = -s$. Dieser Wert wird __Bias__ genannt. Er steht weiterhin für die _Empfindlichkeit_ des Neurons, bei gegebenen Eingabewerten zu feuern.

#### Training eines Perzeptrons

Beim Training des Perzeptrons werden die Testeingabedaten an die Eingabe-Kanäle angelegt. Entspricht die Ausgabe dem erwarteten Ergebnis, so bleiben die Gewichte und der Bias unverändert. Bei einem abweichenden Ergebnis werden die Gewichte nach oben bzw. nach unten korrigiert, d.h. es ist

\begin{equation}
\begin{split}
w_{1,2} &\rightarrow  w_{1,2} + \eta \, x_{1,2}\,,\,  b \rightarrow  b - \eta \;\;\text{(falls 1 erwartet, aber Output 0)}\\
w_{1,2} &\rightarrow  w_{1,2} - \eta  \, x_{1,2}\,,\, b \rightarrow  b - \eta \;\;\text{(falls 0 erwartet, aber Output 1)}\\
\end{split}
\end{equation}

ansonsten bleiben die Gewichte unverändert. Hierbei sind $x_{1,2}$ und  $y$ die Testdaten und $\eta \ll 1$ die Lernrate. Ist $o$ der Output des Perzeptrons, so lässt sich dies vereinfachen zu

$$
\begin{equation}
\begin{split}
w_{1,2} &\rightarrow w_{1,2} + \eta \, (y - o) x_{1,2}\\
b &\rightarrow b + \eta \, (y -o )
\end{split}
\end{equation}
$$
da $y - o =0\;\; \text{oder} \pm 1$ ist.

### Eine Implementierung des Perzeptron

Das Perzeptron ist also ein selbstlernendes Neuron, das gemäß der Perzeptron-Regel funktioniert, ähnlich dem   McCulloch-Pitts-Neuron. Die Hebbsche Lernregel lässt sich sehr einfach in Python-Code übersetzen, denn die booleschen Werte True und False entsprechen den numerischen Werten 1 bzw. 0.

In [8]:
class Perceptron:
    def __init__(self) -> None:
        self.w1 = self.w2 = self.b = 0.
        
    def activation(self, x1: bool, x2: bool) -> bool: 
        return self.w1 * x1 + self.w2 * x2 + self.b > 0
    
    def train(self,input: list[(bool,bool)],target: list[bool],eta:float = 0.1,epochs: int = 1) ->  None:
        """ 
        Trainiert das Perzeptron mit den übergebenen Daten input und den Zielwerten target gemäß dem Lernalgorithmus. 
        eta bestimmt die Lernrate und epochs die Anzahl der Durchläufe. 
        """
        for _ in range(epochs):
            for x, t in zip(input,target): 
                self.hebb(*x, t, eta) 
        
    def hebb(self, x1, x2, y, eta):
        """ Implementierung der Hebbschen Lernregel """
        o = self.activation(x1, x2)  
        self.w1 += eta * (y - o) * x1
        self.w2 += eta * (y - o) * x2        
        self.b  += eta * (y - o) 


### Training und Test unseres Perzeptrons

Die Testdaten sind recht einfach und stellen die Wahrheitstafeln für UND, NICHT-UND, ODER und XODER dar:

In [9]:
logic_functions = {"AND": lambda a,b : a and b, 
                   "OR": lambda a,b : a or b, 
                   "NAND": lambda a,b : not (a and b), 
                   "XOR": lambda a,b : a ^ b}

inputs = [(x,y) for x in boolean for y in boolean]

for (name,lf) in logic_functions.items():
    p = Perceptron()
    targets = [lf(x,y) for x,y in inputs ]
    p.train(inputs,targets,epochs = 10)
    print(f'--- {name} ---')
    for x,y in inputs:
        print(f'({x} {y}) \t-> {p.activation(x,y)}')

--- AND ---
(True True) 	-> True
(True False) 	-> False
(False True) 	-> False
(False False) 	-> False
--- OR ---
(True True) 	-> True
(True False) 	-> True
(False True) 	-> True
(False False) 	-> False
--- NAND ---
(True True) 	-> False
(True False) 	-> True
(False True) 	-> True
(False False) 	-> True
--- XOR ---
(True True) 	-> True
(True False) 	-> True
(False True) 	-> False
(False False) 	-> False


Wie man sieht, gibt es bei der XOR-Verknüpfung kein zufriedenstellendes Ergebnis. Eine Analyse zeigt, dass das einfache zweischichtige Perzeptron diese Aufgabe grundsätzlich nicht lösen kann:

#### XOR lässt sich durch das Perzeptron nicht darstellen

Die XOR-Verknüpfung ist genau dann 1, wenn nur einer der beiden Eingabe-Kanäle aktiv sind. Sind beide oder keiner aktiv, so ergibt die Verknüpfung 0. Was bedeutet das für unser Perzeptron?

Damit das Perzeptron bei der Eingabe von (0,0) 0 ergibt, muss der Bias $b \le 0$ sein, denn aus der Perzeptron-Regel folgt $w_1 \times 0 + w_2 \times 0 + b = b \le 0$. Betrachten wir nun die Eingabe (1,0), bei der das Perzeptron aktiv sein soll. Dies bedeutet $ w_1 + b > 0$, und damit muss $w_1 > -b$ sein. Derselbe Schluss gilt für $w_2$, also $w_2 > -b$. Damit folgt für die Eingabe von (1,1): $w_1 + w_2 + b  < b \le 0$, und damit kann das Perzeptron hier nicht aktiv sein. 

__Die XOR-Verknüpfung ist grundsätzlich mit dem einfachen Perzeptron nicht darstellbar__.

### Linearer Separator

Sind etwa bei der UND- und ODER-Verknüpfung die Eingabedaten in der Ebene durch eine Gerade trennbar, so ist dies bei der XODER-Verknüpfung nicht der Fall. Aufgrund der Tatsache, dass ausschließlich lineare Funktionen verwendet werden, scheitert das Perzeptron an dieser scheinbar einfachen Aufgabe.

Umgekehrt ist aber die NICHT-UND-Aufgabe gelöst, d.h. das Perzeptron kann ein __NAND-Gatter__ darstellen, und damit grundsätzlich jede logische Schaltung.

### Das XOR-Problem

Tatsächlich führte das XOR-Problem zusammen mit weiteren Schwierigkeiten nach der ersten Euphorie zu einem starken Nachlassen des Interesses an der Erforschung künstlicher Neuronaler Netwerke. Die Lösung des **XOR-Problems** liegt in der Verwendung weiterer Schichten von Neuronen und nicht-linearer Funktionen, und sie führte zur Entwicklung der heute verwendeten Künstlichen Neuronalen Netze.